In [1]:
import numpy as np
import pandas as pd

import statsmodels.api as sm
from sklearn.metrics import r2_score, mean_squared_error

df = pd.read_excel("popular_times_with_restaurant_and_june_taxi_demand.xlsx")

print(df.shape)
df.head()

C:\Anaconda\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


(41882, 24)


,restaurant_id,original_name,original_address,matched_name,google_id,matched_address,latitude,longitude,day,hour,...,weekday,demand_month,taxi_zone_id,taxi_zone_name,estimated_area_sqft,capacity,turnover_rate,takeaway_ratio,eat_in_ratio,dropoff_count
0,50180332,HANI'S BAKERY & CAFE,"67 COOPER SQUARE, MANHATTAN, NY 10003",Hani’s bakery + café,0x89c2590068922807:0x22ca1ea7cc532a1a,NaN,40.729099,-73.989937,7,6,...,6,6,79,East Village,1272.5,76.35,1.1,0.2,0.8,63.0
1,50180332,HANI'S BAKERY & CAFE,"67 COOPER SQUARE, MANHATTAN, NY 10003",Hani’s bakery + café,0x89c2590068922807:0x22ca1ea7cc532a1a,NaN,40.729099,-73.989937,7,7,...,6,6,79,East Village,1272.5,76.35,1.1,0.2,0.8,63.0
2,50180332,HANI'S BAKERY & CAFE,"67 COOPER SQUARE, MANHATTAN, NY 10003",Hani’s bakery + café,0x89c2590068922807:0x22ca1ea7cc532a1a,NaN,40.729099,-73.989937,7,8,...,6,6,79,East Village,1272.5,76.35,1.1,0.2,0.8,64.5
3,50180332,HANI'S BAKERY & CAFE,"67 COOPER SQUARE, MANHATTAN, NY 10003",Hani’s bakery + café,0x89c2590068922807:0x22ca1ea7cc532a1a,NaN,40.729099,-73.989937,7,9,...,6,6,79,East Village,1272.5,76.35,1.1,0.2,0.8,259.0
4,50180332,HANI'S BAKERY & CAFE,"67 COOPER SQUARE, MANHATTAN, NY 10003",Hani’s bakery + café,0x89c2590068922807:0x22ca1ea7cc532a1a,NaN,40.729099,-73.989937,7,10,...,6,6,79,East Village,1272.5,76.35,1.1,0.2,0.8,435.0


In [2]:
import re

def parse_typical_time(x):
    if pd.isna(x):
        return np.nan

    x = str(x).lower().strip()

    # e.g.：People typically spend 15 min to 1 hr here
    match = re.search(
        r"spend\s+(\d+(?:\.\d+)?)\s*(min|mins|minute|minutes|hr|hrs|hour|hours)"
        r"(?:\s+to\s+(\d+(?:\.\d+)?)\s*(min|mins|minute|minutes|hr|hrs|hour|hours))?",
        x
    )

    if not match:
        return np.nan

    v1 = float(match.group(1))
    u1 = match.group(2)

    v2 = match.group(3)
    u2 = match.group(4)

    def to_minutes(value, unit):
        if unit in ["hr", "hrs", "hour", "hours"]:
            return value * 60
        elif unit in ["min", "mins", "minute", "minutes"]:
            return value
        else:
            return np.nan

    t1 = to_minutes(v1, u1)

    if v2 is not None:
        t2 = to_minutes(float(v2), u2)
        return (t1 + t2) / 2
    else:
        return t1

df["typical_time_mid"] = df["typical_time_spent"].apply(parse_typical_time)

In [3]:
df[["typical_time_spent","typical_time_mid"]].head(20)

,typical_time_spent,typical_time_mid
0,People typically spend 15 min to 1 hr here,37.5
1,People typically spend 15 min to 1 hr here,37.5
2,People typically spend 15 min to 1 hr here,37.5
3,People typically spend 15 min to 1 hr here,37.5
4,People typically spend 15 min to 1 hr here,37.5
5,People typically spend 15 min to 1 hr here,37.5
6,People typically spend 15 min to 1 hr here,37.5
7,People typically spend 15 min to 1 hr here,37.5
8,People typically spend 15 min to 1 hr here,37.5
9,People typically spend 15 min to 1 hr here,37.5


In [4]:
df["popularity"] = (
    df["rating"]
    * np.log1p(df["reviews"])
)

In [5]:
df["hour_sin"] = np.sin(
    2*np.pi*df["hour"]/24
)

df["hour_cos"] = np.cos(
    2*np.pi*df["hour"]/24
)

In [6]:
df["friday"] = (df["day"]==5).astype(int)

df["saturday"] = (df["day"]==6).astype(int)

df["sunday"] = (df["day"]==7).astype(int)

In [7]:
df["ln_area"] = np.log(df["estimated_area_sqft"])

In [8]:
df["ln_dropoff"] = np.log1p(df["dropoff_count"])

In [9]:
df["ln_reviews"] = np.log1p(df["reviews"])

In [10]:
restaurant_ids = df["restaurant_id"].unique()

print(len(restaurant_ids))

323


In [11]:
train_restaurants = restaurant_ids[:155]

test_restaurants = restaurant_ids[155:]

In [12]:
train_df = df[df["restaurant_id"].isin(train_restaurants)].copy()

test_df = df[df["restaurant_id"].isin(test_restaurants)].copy()

print(train_df.shape)
print(test_df.shape)

(20312, 34)
(21570, 34)


In [13]:
feature_cols = [
    "hour_sin",
    "hour_cos",
    "friday",
    "saturday",
    "sunday",
    "typical_time_mid",
    "rating",
    "ln_reviews",
]

target_col = "busyness_score"

train_model_df = train_df[
    feature_cols + [target_col]
].replace([np.inf, -np.inf], np.nan).dropna()

test_model_df = test_df[
    feature_cols + [target_col]
].replace([np.inf, -np.inf], np.nan).dropna()

In [14]:
X_train = train_model_df[feature_cols]
X_train = sm.add_constant(X_train)

y_train = train_model_df[target_col]

X_test = test_model_df[feature_cols]
X_test = sm.add_constant(X_test)

y_test = test_model_df[target_col]

model = sm.OLS(y_train, X_train)
results = model.fit()

print(results.summary())

                            OLS Regression Results                            
Dep. Variable:         busyness_score   R-squared:                       0.285
Model:                            OLS   Adj. R-squared:                  0.285
Method:                 Least Squares   F-statistic:                     553.5
Date:                Thu, 02 Jul 2026   Prob (F-statistic):               0.00
Time:                        00:18:37   Log-Likelihood:                -51590.
No. Observations:               11105   AIC:                         1.032e+05
Df Residuals:                   11096   BIC:                         1.033e+05
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
const               68.8498      2.570  

In [15]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

train_pred = results.predict(X_train)
test_pred = results.predict(X_test)

print("Train")
print("R2 =", r2_score(y_train, train_pred))
print("MAE =", mean_absolute_error(y_train, train_pred))
print("RMSE =", np.sqrt(mean_squared_error(y_train, train_pred)))

print()

print("Test")
print("R2 =", r2_score(y_test, test_pred))
print("MAE =", mean_absolute_error(y_test, test_pred))
print("RMSE =", np.sqrt(mean_squared_error(y_test, test_pred)))

Train
R2 = 0.2852203219540693
MAE = 20.860491826815263
RMSE = 25.19581978221319

Test
R2 = 0.29072650213162654
MAE = 20.96753450992686
RMSE = 25.251973740579647


In [16]:
feature_cols = [
    "hour_sin",
    "hour_cos",
    "friday",
    "saturday",
    "sunday",
    "typical_time_mid",
    "rating",
    "ln_reviews",
    "ln_area",
    "takeaway_ratio"
]

target_col = "busyness_score"

train_model_df = train_df[
    feature_cols + [target_col]
].replace([np.inf, -np.inf], np.nan).dropna()

test_model_df = test_df[
    feature_cols + [target_col]
].replace([np.inf, -np.inf], np.nan).dropna()

In [17]:
X_train = train_model_df[feature_cols]
X_train = sm.add_constant(X_train)

y_train = train_model_df[target_col]

X_test = test_model_df[feature_cols]
X_test = sm.add_constant(X_test)

y_test = test_model_df[target_col]

model = sm.OLS(y_train, X_train)
results = model.fit()

print(results.summary())

                            OLS Regression Results                            
Dep. Variable:         busyness_score   R-squared:                       0.287
Model:                            OLS   Adj. R-squared:                  0.287
Method:                 Least Squares   F-statistic:                     447.3
Date:                Thu, 02 Jul 2026   Prob (F-statistic):               0.00
Time:                        00:18:37   Log-Likelihood:                -51573.
No. Observations:               11105   AIC:                         1.032e+05
Df Residuals:                   11094   BIC:                         1.032e+05
Df Model:                          10                                         
Covariance Type:            nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
const               58.4489      3.181  

In [18]:
train_pred = results.predict(X_train)
test_pred = results.predict(X_test)

print("Train")
print("R2 =", r2_score(y_train, train_pred))
print("MAE =", mean_absolute_error(y_train, train_pred))
print("RMSE =", np.sqrt(mean_squared_error(y_train, train_pred)))

print()

print("Test")
print("R2 =", r2_score(y_test, test_pred))
print("MAE =", mean_absolute_error(y_test, test_pred))
print("RMSE =", np.sqrt(mean_squared_error(y_test, test_pred)))

Train
R2 = 0.287347769019322
MAE = 20.762417619315386
RMSE = 25.158295829479034

Test
R2 = 0.29717184433915944
MAE = 20.873343995916848
RMSE = 25.136976453302143


In [19]:
feature_cols = [
    "hour_sin",
    "hour_cos",
    "friday",
    "saturday",
    "sunday",
    "typical_time_mid",
    "rating",
    "ln_reviews",
    "ln_dropoff",
    "ln_area",
    "takeaway_ratio"  
]


target_col = "busyness_score"

train_model_df = train_df[
    feature_cols + [target_col]
].replace([np.inf, -np.inf], np.nan).dropna()

test_model_df = test_df[
    feature_cols + [target_col]
].replace([np.inf, -np.inf], np.nan).dropna()

In [20]:
X_train = train_model_df[feature_cols]
X_train = sm.add_constant(X_train)

y_train = train_model_df[target_col]

X_test = test_model_df[feature_cols]
X_test = sm.add_constant(X_test)

y_test = test_model_df[target_col]

model = sm.OLS(y_train, X_train)
results = model.fit()

print(results.summary())

                            OLS Regression Results                            
Dep. Variable:         busyness_score   R-squared:                       0.287
Model:                            OLS   Adj. R-squared:                  0.287
Method:                 Least Squares   F-statistic:                     406.7
Date:                Thu, 02 Jul 2026   Prob (F-statistic):               0.00
Time:                        00:18:37   Log-Likelihood:                -51573.
No. Observations:               11105   AIC:                         1.032e+05
Df Residuals:                   11093   BIC:                         1.033e+05
Df Model:                          11                                         
Covariance Type:            nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
const               57.3649      3.442  

In [21]:
train_pred = results.predict(X_train)
test_pred = results.predict(X_test)

print("Train")
print("R2 =", r2_score(y_train, train_pred))
print("MAE =", mean_absolute_error(y_train, train_pred))
print("RMSE =", np.sqrt(mean_squared_error(y_train, train_pred)))

print()

print("Test")
print("R2 =", r2_score(y_test, test_pred))
print("MAE =", mean_absolute_error(y_test, test_pred))
print("RMSE =", np.sqrt(mean_squared_error(y_test, test_pred)))

Train
R2 = 0.28739136836746215
MAE = 20.762887375091484
RMSE = 25.157526238065515

Test
R2 = 0.29733747246414977
MAE = 20.87038512846562
RMSE = 25.134014395252205


In [44]:
features = [
    "hour_sin",
    "hour_cos",
    "friday",
    "saturday",
    "sunday",
    "typical_time_mid",
    "rating",
    "ln_reviews",
    "ln_dropoff",
    "ln_area",
    "takeaway_ratio"
]

In [46]:
X_train = train_model_df[features]
y_train = train_model_df["busyness_score"]

X_test = test_model_df[features]
y_test = test_model_df["busyness_score"]

In [48]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=500,
    max_depth=12,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

train_pred = rf.predict(X_train)
test_pred = rf.predict(X_test)

In [50]:
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)

print("Train")
print(r2_score(y_train, train_pred))

print(np.sqrt(mean_squared_error(y_train, train_pred)))

print(mean_absolute_error(y_train, train_pred))

print()

print("Test")
print(r2_score(y_test, test_pred))

print(np.sqrt(mean_squared_error(y_test, test_pred)))

print(mean_absolute_error(y_test, test_pred))

Train
0.8283902284241663
12.345634113116775
8.478026752783324

Test
0.28996688139516
25.265492354916578
19.49522459112057


In [52]:
importance = (
    pd.Series(
        rf.feature_importances_,
        index=features
    )
    .sort_values(ascending=False)
)

print(importance)

hour_sin            0.245329
hour_cos            0.188281
takeaway_ratio      0.147828
ln_area             0.120229
ln_dropoff          0.077428
ln_reviews          0.073782
typical_time_mid    0.070241
rating              0.058801
saturday            0.007591
sunday              0.006314
friday              0.004177
dtype: float64


In [54]:
pip install xgboost

In [56]:
from xgboost import XGBRegressor

xgb = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb.fit(X_train, y_train)

train_pred = xgb.predict(X_train)
test_pred = xgb.predict(X_test)

In [58]:
print("Train")
print(r2_score(y_train, train_pred))

print(np.sqrt(mean_squared_error(y_train, train_pred)))

print(mean_absolute_error(y_train, train_pred))

print()

print("Test")
print(r2_score(y_test, test_pred))

print(np.sqrt(mean_squared_error(y_test, test_pred)))

print(mean_absolute_error(y_test, test_pred))

Train
0.9202253222465515
8.417340645070523
6.147100925445557

Test
0.29785358905792236
25.12478258029373
19.548452377319336


In [60]:
importance = (
    pd.Series(
        xgb.feature_importances_,
        index=features
    )
    .sort_values(ascending=False)
)

print(importance)

takeaway_ratio      0.198865
hour_sin            0.163393
hour_cos            0.146722
ln_area             0.117812
typical_time_mid    0.087035
rating              0.084038
ln_reviews          0.075738
ln_dropoff          0.038124
saturday            0.030828
friday              0.029241
sunday              0.028206
dtype: float32
